In [1]:
import numpy as np

In [2]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

# Run all the elevation planes
# - Elevation ... elevation in the frontal plane
# - Scabduction ... elevation in the scapular plane
# - Flexion ... elevation in the sagittal plane

participant = ['par1','par2','par3']
motion_list  =  ['Elevation','Scabduction','Flexion']

GH_seq = 'YZY' 

act_w = 1

In [3]:
clav_pos = 0.4
tilt_y = 13
tilt_z = -6.5
w_traj = 200
quat_constraints = [0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15]

for ipar in range(len(participant)):
    for imot in range(len(motion_list)):
        OS_struct = sc.io.loadmat('../Motions/'+participant[ipar]+'/OS_model.mat')
        q,u,fr,frstar,kinematical = eq.create_eoms_quat_no_RF(OS_struct,hand_weight = 0,derive = 'numeric',gen_matlab_functions = 0)

        TE,activations,TE_conoid, _,_,_,_,_, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = None, derive = 'numeric', RC_lim = 1.0)
        include_activation_dynamics = True

        struct_name = 'res_quat_'+motion_list[imot]+'_'+str(int(w_traj))
        eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]))

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            for i in range(len(activations)):
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))

            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)

        interval_value = 0.04
        file = '../Motions/' + participant[ipar] + '/' + motion_list[imot] + '/' + motion_list[imot]
        traj_original, omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
        q0_t0 = traj_original[:,0][:4]
        traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)

        
        indexes_clav_scap = 1
        indexes_hum = 1

        if include_activation_dynamics:
            state_symbols = tuple(q+u+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols) 
        num_q = len(q)
        num_u = len(u)
        num_faux = 0
        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        
        objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.objective_traj_quat(num_q,interval_value,clav_pos,True)

        objective_act,objective_act_jac = eq.objective_min_activation(activations,interval_value)

        obj_min_diff,obj_min_diff_jac = eq.objective_state_diff(num_nodes,interval_value)

        w_diff_exc = 1e-3
        w_diff_vel = 1e-3


        def obj(free):
            min_traj = w_traj * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            min_SC_t0 = w_traj * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))

            min_torque = act_w * np.sum(objective_act(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))

            obj = (min_traj + min_vel_dif + min_torque + min_SC_t0) #   
            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
                obj += (min_exc_dif)

            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)
            grad[:num_q*num_nodes] += w_traj * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            grad[0::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))

            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] += act_w * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))


            if include_activation_dynamics:
                grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))

            return grad

        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01)))

        instance_constraints = []
        instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
        instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
        instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH

            
        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)
        
        for i in range(num_q):
            bndrs.update({q[i]: (min(traj_original[i,:])-0.15, max(traj_original[i,:])+0.15)})

        if participant[ipar] == 'par2' and motion_list[imot] == 'Scabduction':
            for i in range(num_u):
                bndrs.update({u[i]: (-3.0, 3.0)})

        print(bndrs)

        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
                    parallel = False)


        time_to_create = tm.time() - start
        print(time_to_create)

        prob.add_option('limited_memory_max_history', 40)
        initial_guess = np.ones(prob.num_free)*0.0

        time_2_solve_start = tm.time()

        prob.add_option('max_iter',2000)
        initial_guess[:13*num_nodes] = traj_original.flatten()
        initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()
  
        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)

        file_name = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name+'.mat'

        tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

        file_name_mot = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mot'
        tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)

obj_check 1849.2517170392287
obj_grad_check -633.9710761464651
{act_97(t): (0.0, 1.0), act_96(t): (0.0, 1.0), act_79(t): (0.0, 1.0), act_82(t): (0.0, 1.0), act_38(t): (0.0, 1.0), act_113(t): (0.0, 1.0), act_57(t): (0.0, 1.0), act_22(t): (0.0, 1.0), act_13(t): (0.0, 1.0), act_78(t): (0.0, 1.0), act_102(t): (0.0, 1.0), act_72(t): (0.0, 1.0), act_69(t): (0.0, 1.0), act_117(t): (0.0, 1.0), act_73(t): (0.0, 1.0), act_26(t): (0.0, 1.0), act_19(t): (0.0, 1.0), act_46(t): (0.0, 1.0), act_35(t): (0.0, 1.0), act_98(t): (0.0, 1.0), act_42(t): (0.0, 1.0), act_108(t): (0.0, 1.0), act_61(t): (0.0, 1.0), act_115(t): (0.0, 1.0), act_7(t): (0.0, 1.0), act_40(t): (0.0, 1.0), act_55(t): (0.0, 1.0), act_52(t): (0.0, 1.0), act_70(t): (0.0, 1.0), act_94(t): (0.0, 1.0), act_43(t): (0.0, 1.0), act_5(t): (0.0, 1.0), act_93(t): (0.0, 1.0), act_65(t): (0.0, 1.0), act_33(t): (0.0, 1.0), act_60(t): (0.0, 1.0), act_3(t): (0.0, 1.0), act_111(t): (0.0, 1.0), act_134(t): (0.0, 1.0), act_136(t): (0.0, 1.0), act_133(t):